# Занятие 11. Свёртка как матричное умножение

Разберем две записи одной операции: $y=A(K)x$ и $y=P(X)k$. Первая матрица содержит повторяющиеся веса ядра, вторая — развернутые окна входа. Затем проверим, как через эти матрицы проходят градиенты по входу и общим весам.

Во всех численных примерах вход $X$ имеет размер $4\times4$, ядро — $2\times2$, шаг равен 1, padding отсутствует, смещение равно нулю. Выход имеет размер $3\times3$:

$$Y_{ij}=\sum_{u=0}^1\sum_{v=0}^1K_{uv}X_{i+u,j+v}.$$

Код и исходные данные приведены ниже целиком. Для графиков нужна подпапка `support/` из комплекта занятия.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from support.week11_utils import draw_matrices, show_table

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

from support.lesson11_visuals import setup_style
setup_style()

In [ ]:
X = np.arange(1, 17, dtype=float).reshape(4, 4)
print("Вход X:")
print(X)

## 1. Свертка как умножение на большую разреженную матрицу

Для фиксированного ядра операция линейна по входу. Из исходной формулы
$Y_{ij}=\sum_{u,v}K_{uv}X_{i+u,j+v}$ переходят к нумерации пикселей по строкам:
$x=\operatorname{vec}(X)\in\mathbb R^{16}$, $y=\operatorname{vec}(Y)\in\mathbb R^9$.
Для первого выхода:

$$y_0=K_{00}x_0+K_{01}x_1+K_{10}x_4+K_{11}x_5.$$

Следовательно, первая строка матрицы $A(K)\in\mathbb R^{9\times16}$ содержит
$K_{00},K_{01},K_{10},K_{11}$ в столбцах $0,1,4,5$ и нули в остальных.
Для следующего окна эти коэффициенты сдвигают на один столбец; при переходе
на новую строку изображения учитывают его ширину. Получают

$$y=A(K)x+b\mathbf1.$$

Это блочно-тёплицева структура: строки состоят из сдвинутых повторений одного
ядра. У матрицы 144 позиции, но обучают только **четыре** числа ядра и одно
смещение. Повторенные элементы — один параметр, а не независимые веса.

In [ ]:
# Для наглядности все четыре коэффициента различны и ненулевые.
K_matrix = np.array([[1., 2.], [3., 4.]])
A = np.zeros((9, 16))
patches = []
for row in range(3):
    for column in range(3):
        output_index = row * 3 + column
        patch = X[row:row + 2, column:column + 2]
        patches.append(patch.reshape(-1))
        for u in range(2):
            for v in range(2):
                input_index = (row + u) * 4 + column + v
                A[output_index, input_index] = K_matrix[u, v]

matrix_result = A @ X.reshape(-1)
figure, axis = plt.subplots(figsize=(12, 5))
image = axis.imshow(A, cmap="Blues", vmin=0, vmax=4)
for (i, j), value in np.ndenumerate(A):
    axis.text(j, i, f"{value:g}", ha="center", va="center", fontsize=9, color="white" if value >= 3 else "black")
axis.set(xlabel="номер входного пикселя", ylabel="номер выходного пикселя",
         title="A(K): повторение четырех весов, остальные связи отсутствуют",
         xticks=np.arange(16), yticks=np.arange(9))
figure.tight_layout()
plt.show()

### Окно изображения и строка матрицы — один шаг

Передвиньте слайдер от 0 до 8. Оранжевая рамка одновременно выбирает окно X, строку A(K) и элемент Y. Четыре ненулевых коэффициента строки — те же веса ядра; меняются только их позиции. Над рисунком показана сумма четырёх произведений для выбранного выхода.

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
from support.lesson11_visuals import export_matrix_windows

window_plot = export_matrix_windows(X, K_matrix, Path("outputs/notes_11_matrix_windows.html"))
display(FileLink("outputs/notes_11_matrix_windows.html"))

## 2. Другая матричная запись: im2col

Ту же формулу $Y_{ij}=\sum_{u,v}K_{uv}X_{i+u,j+v}$ можно переписать, считая
не вход, а веса вектором. Каждое окно разворачивают в строку:

$$P(X)=\begin{pmatrix}1&2&5&6\\2&3&6&7\\3&4&7&8\\\vdots&\vdots&\vdots&\vdots\end{pmatrix},
\qquad k=\operatorname{vec}(K),\qquad y=P(X)k.$$

Для этого примера $P$ имеет размер $9\times4$. При нескольких выходных каналах
ядра размещают в столбцах матрицы $W$, и сразу получают $Y_{\mathrm{flat}}=PW$.

Две записи описывают **один результат**: $A(K)x=P(X)k$.
Матрица $A$ содержит веса, матрица $P$ — повторяющиеся пиксели входа.
`Unfold` в PyTorch располагает окна в столбцах; для записи выше результат транспонируют.
Реальные библиотеки могут использовать im2col, прямой алгоритм и другие реализации;
`Conv2d` не обязан явно хранить полную матрицу $A$ или $P$.

In [ ]:
P = np.stack(patches)
im2col_result = P @ K_matrix.reshape(-1)
tensor_X = torch.tensor(X, dtype=torch.float64)[None, None]
unfold = nn.Unfold(kernel_size=2)
torch_patches = unfold(tensor_X)[0].T.numpy()
assert np.array_equal(P, torch_patches)
assert np.allclose(matrix_result, im2col_result)

convolution = nn.Conv2d(1, 1, 2, bias=False).double()
with torch.no_grad():
    convolution.weight.copy_(torch.tensor(K_matrix)[None, None])
torch_result = convolution(tensor_X).detach().numpy().reshape(-1)
assert np.allclose(torch_result, matrix_result)
show_table(pd.DataFrame(P, columns=["верх-лево", "верх-право", "низ-лево", "низ-право"]))

In [ ]:
# В каждой строке P лежит одно окно; цвет показывает повторяющиеся пиксели.
draw_matrices([X, P, K_matrix.reshape(-1, 1), im2col_result.reshape(3, 3)],
              ["X: 4x4", "P(X): 9 окон x 4 пикселя", "k: 4x1", "P @ k: 3x3"])

## 3. Как проходит градиент через общие веса

Для линейной операции $y=A(K)x$ обозначают входящий градиент через
$g=\partial L/\partial y$. Дифференциал по входу:

$$dL=g^Tdy=g^TA\,dx=(A^Tg)^Tdx,
\qquad \frac{\partial L}{\partial x}=A^Tg.$$

По весам используют другую запись, $y=P(X)k$:

$$dL=g^TP\,dk=(P^Tg)^Tdk,
\qquad\frac{\partial L}{\partial k}=P^Tg.$$

Один коэффициент ядра используется во всех окнах, поэтому его градиент — **сумма
вкладов всех положений**, а не градиент одного окна. Для батча добавляется сумма
по изображениям. При среднем loss масштаб $1/N$ уже входит в $g$.
Транспонированная свертка реализует линейную операцию обратного прохода по входу;
она в общем случае не является обратной матрицей и не восстанавливает исходное изображение.

In [ ]:
g = torch.arange(1, 10, dtype=torch.float64).reshape(1, 1, 3, 3)
tensor_X.requires_grad_(True)
output = convolution(tensor_X)
loss = (output * g).sum()
loss.backward()
expected_dx = A.T @ g.numpy().reshape(-1)
expected_dk = P.T @ g.numpy().reshape(-1)
assert np.allclose(tensor_X.grad.numpy().reshape(-1), expected_dx)
assert np.allclose(convolution.weight.grad.numpy().reshape(-1), expected_dk)
print("Градиенты по входу и общим весам совпали с матричным расчетом")

In [ ]:
# Показывают градиенты, вычисленные выше, а не условные стрелки.
grid_g = g.numpy()[0, 0]
grid_dx = tensor_X.grad.numpy()[0, 0]
grid_dk = convolution.weight.grad.numpy()[0, 0]
draw_matrices([grid_g, grid_dx, grid_dk],
              ["Входящий g: 3x3", "A.T @ g: вклад всех окон во вход", "P.T @ g: сумма по положениям"])

# Один общий коэффициент K[0,0] получает девять слагаемых.
contributions = P[:, 0] * grid_g.reshape(-1)
figure, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
positions = np.arange(len(contributions))
axes[0].bar(positions, contributions, color="#087f8c", alpha=.8)
axes[0].set(xlabel="Номер окна", ylabel="Одно слагаемое", title="Каждое окно использует тот же K[0,0]")
cumulative = np.r_[0, np.cumsum(contributions)]
axes[1].step(np.arange(10), cumulative, where="post", color="#7963ad")
axes[1].scatter(np.arange(10), cumulative, color="#7963ad", s=25)
axes[1].annotate(f"dL/dK[0,0] = {cumulative[-1]:g}", (9, cumulative[-1]),
                 xytext=(-12, -27), textcoords="offset points", ha="right", weight="bold")
axes[1].set(xlabel="Сколько окон учтено", ylabel="Накопленная сумма", title="Градиенты по общему весу складываются")
for axis in axes:
    axis.grid(axis="y")
plt.show()
assert np.isclose(contributions.sum(), grid_dk[0, 0])